In [3]:
"""
Ячейка 2: Парсинг интента из колонки response
"""

from datasets import load_dataset
import pandas as pd
import re

ds2 = load_dataset("ikshana-zehntech/ecommerce_intent_recognition")
ikshana_df = pd.DataFrame(ds2['train'])

# Парсинг интента из response
# Формат: "intent: cancel_order, variable name: order_number, variable_value: 2B4FZM0C"

def parse_intent(response_text):
    """Извлекает интент из строки response"""
    match = re.search(r'intent:\s*(\w+)', str(response_text))
    if match:
        return match.group(1)
    return None

ikshana_df['intent'] = ikshana_df[' response'].apply(parse_intent)

# Смотрим какие интенты есть
print(f"📊 ikshana: {len(ikshana_df)} строк")
print(f"Уникальных интентов: {ikshana_df['intent'].nunique()}")
print(f"\nИнтенты:")
print(ikshana_df['intent'].value_counts().to_string())

📊 ikshana: 27869 строк
Уникальных интентов: 27

Интенты:
intent
change_order                1994
edit_account                1000
switch_account              1000
check_invoice               1000
contact_customer_service    1000
complaint                   1000
contact_human_agent          999
delivery_period              999
get_invoice                  999
newsletter_subscription      999
check_payment_methods        999
registration_problems        999
payment_issue                999
cancel_order                 998
place_order                  998
track_refund                 998
set_up_shipping_address      997
get_refund                   997
check_refund_policy          997
create_account               997
review                       997
delivery_options             995
delete_account               995
track_order                  995
recover_password             995
change_shipping_address      973
check_cancellation_fee       950


In [7]:
"""
Ячейка 3: Маппинг интентов ikshana → наши 10 категорий
"""

# Маппинг (дополни после того как увидишь полный список интентов)
IKSHANA_INTENT_MAP = {
    # ── order_status ──
    "cancel_order":            "order_status",
    "track_order":             "order_status",
    "order_status":            "order_status",
    "change_order":            "order_status",
    "place_order":             "order_status",
    "check_cancellation_fee":  "order_status",
    
    # ── payment_refund ──
    "payment_issue":           "payment_refund",
    "refund":                  "payment_refund",
    "payment_method":          "payment_refund",
    "check_invoice":           "payment_refund",
    "get_invoice":             "payment_refund",
    "check_payment_methods":   "payment_refund",
    "track_refund":            "payment_refund",
    "get_refund":              "payment_refund",
    "check_refund_policy":     "payment_refund",
    
    # ── return_exchange ──
    "return_product":          "return_exchange",
    "exchange_product":        "return_exchange",
    "damaged_product":         "return_exchange",
    
    # ── product_info ──
    "product_inquiry":         "product_info",
    "product_availability":    "product_info",
    "review":                  "product_info",
    
    # ── delivery ──
    "delivery_status":         "delivery",
    "shipping_info":           "delivery",
    "delivery_period":         "delivery",
    "delivery_options":        "delivery",
    "set_up_shipping_address": "delivery",
    "change_shipping_address": "delivery",
    
    # ── account ──
    "account_issue":           "account",
    "reset_password":          "account",
    "create_account":          "account",
    "edit_account":            "account",
    "switch_account":          "account",
    "delete_account":          "account",
    "recover_password":        "account",
    "registration_problems":   "account",
    "newsletter_subscription": "account",
    
    # ── general_info ──
    "store_info":              "general_info",
    "contact_info":            "general_info",
    "contact_customer_service":"general_info",
    
    # ── other ──
    "feedback":                "other",
    "human_agent":             "other",
    "contact_human_agent":     "other",
    "complaint":               "other",
}

ikshana_df['our_category'] = ikshana_df['intent'].map(IKSHANA_INTENT_MAP)

unmapped = ikshana_df[ikshana_df['our_category'].isna()]
if len(unmapped) > 0:
    print(f"⚠️ Ещё незамапленные: {unmapped['intent'].value_counts().to_string()}")
else:
    print("✅ Все интенты замаплены")

print(f"\n📊 Распределение:")
print(ikshana_df['our_category'].value_counts().to_string())

ikshana_final = ikshana_df[ikshana_df['our_category'].notna()][['context', 'our_category', 'intent']].copy()
ikshana_final.columns = ['text', 'category', 'original_intent']
ikshana_final['source'] = 'ikshana'
ikshana_final['language'] = 'en'

print(f"\n✅ ikshana: {len(ikshana_final)} примеров")

✅ Все интенты замаплены

📊 Распределение:
our_category
payment_refund    6989
account           6985
order_status      5935
delivery          3964
other             1999
general_info      1000
product_info       997

✅ ikshana: 27869 примеров
